# Feature Pipeline Validation



This notebook evaluates the same shared feature-pipeline workflow in either runtime lane.

The active environment decides whether the current run uses the local S3-compatible store or the hosted BigQuery store, but ingest, feature engineering, validation, persistence, and Feast preparation follow the same application code in both cases.

If a summary from the other backend is already available, the notebook also compares the stable output fields in place.


## Evaluation Steps



1. Load the active runtime bindings and detect the current storage backend.

2. Fetch forecast data for one configured spot.

3. Engineer the curated feature frame.

4. Run the validation contract.

5. Persist and reload the curated frame through the configured backend.

6. Project the stored dataset into the Feast offline and entity-row shapes.

7. Emit a backend-tagged review summary.

8. Compare the current run with the other backend summary when that artifact is available.


In [ ]:
from pathlib import Path
import json
import os

import pandas as pd
from IPython.display import display

from foehncast import config as config_module
from foehncast.config import get_api_config, get_spots, get_storage_config
from foehncast.feature_pipeline.engineer import engineer_features
from foehncast.feature_pipeline.feast import (
    build_entity_rows,
    build_offline_store_frame,
    export_offline_store,
)
from foehncast.feature_pipeline.ingest import fetch_forecast
from foehncast.feature_pipeline.store import read_features, write_features
from foehncast.feature_pipeline.validate import run_validation
from foehncast.feast_runtime import resolve_runtime_config
from foehncast.paths import feast_offline_path, project_root as foehncast_project_root


def load_env_file(path: Path) -> None:
    if not path.exists():
        return

    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line.removeprefix("export ").strip()
        if "=" not in line:
            continue

        name, value = line.split("=", 1)
        name = name.strip()
        value = value.strip().strip('"').strip("'")
        if name:
            os.environ.setdefault(name, value)


load_env_file(foehncast_project_root() / ".env")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

In [ ]:
spots = {spot["id"]: spot for spot in get_spots()}
spot_id = "silvaplana"
dataset = "notebook_eval"

spot = spots[spot_id]
api_cfg = get_api_config()["open_meteo"]
storage_cfg = get_storage_config()
project_root = foehncast_project_root()
summary_dir = project_root / ".state" / "notebook_reviews"
summary_dir.mkdir(parents=True, exist_ok=True)
backend = storage_cfg["backend"]
runtime_lane = (
    "cloud" if backend == "bigquery" else "local" if backend == "s3" else "unknown"
)
summary_path = summary_dir / f"feature_pipeline_summary_{backend}.json"
base_export_path = feast_offline_path(dataset)
feast_export_path = base_export_path.with_name(
    f"{base_export_path.stem}_{backend}{base_export_path.suffix}"
)

if backend == "bigquery":
    storage_target = ".".join(
        filter(
            None,
            [
                storage_cfg.get("bigquery_project_id", ""),
                storage_cfg.get("bigquery_dataset", ""),
                storage_cfg.get("bigquery_table", ""),
            ],
        )
    )
elif backend == "s3":
    storage_target = f"s3://{storage_cfg['s3_bucket']}/{dataset}/{spot_id}.parquet"
else:
    raise ValueError(f"Unsupported storage backend for notebook review: {backend}")

try:
    feast_runtime_cfg = resolve_runtime_config()
    feast_runtime_status = "ready"
except Exception as exc:
    feast_runtime_cfg = None
    feast_runtime_status = f"error: {exc}"

scope_df = pd.DataFrame(
    [
        {
            "runtime_lane": runtime_lane,
            "spot_id": spot_id,
            "dataset": dataset,
            "storage_backend": backend,
            "storage_target": storage_target,
            "api_forecast_url": api_cfg["forecast_url"],
            "s3_endpoint": storage_cfg.get("s3_endpoint", ""),
            "bigquery_project_id": storage_cfg.get("bigquery_project_id", ""),
            "feast_runtime_status": feast_runtime_status,
            "feast_export_path": str(feast_export_path),
        }
    ]
)

display(scope_df)
print(f"Summary output: {summary_path}")

In [ ]:
raw_df = fetch_forecast(spot["lat"], spot["lon"])

ingest_summary = pd.DataFrame(
    [
        {
            "raw_rows": int(len(raw_df)),
            "raw_columns": int(raw_df.shape[1]),
            "time_min": raw_df.index.min(),
            "time_max": raw_df.index.max(),
        }
    ]
)

display(ingest_summary)
display(raw_df.head())

In [ ]:
feature_df = engineer_features(raw_df, spot["shore_orientation_deg"])

feature_summary = pd.DataFrame(
    [
        {
            "feature_rows": int(len(feature_df)),
            "feature_columns": int(feature_df.shape[1]),
            "missing_values": int(feature_df.isna().sum().sum()),
        }
    ]
)

display(feature_summary)
display(feature_df.head())

In [ ]:
validation_cfg = config_module.get_validation_config()
validation = run_validation(feature_df, spot_id)

validation_summary = pd.DataFrame(
    [
        {
            "is_valid": bool(validation.is_valid),
            "required_columns": len(validation_cfg["required_columns"]),
            "missing_columns": len(validation.missing_columns),
            "range_violations": len(validation.range_violations),
        }
    ]
)

display(validation_summary)
if validation.missing_columns:
    display(pd.DataFrame({"missing_columns": sorted(validation.missing_columns)}))
if len(validation.range_violations) > 0:
    display(pd.DataFrame(validation.range_violations))

In [ ]:
write_features(feature_df, spot_id=spot_id, dataset=dataset)
stored_df = read_features(spot_id=spot_id, dataset=dataset).sort_index()
expected_df = feature_df.sort_index()

numeric_columns = expected_df.select_dtypes(include=["number"]).columns.tolist()
max_numeric_abs_delta = 0.0
if numeric_columns:
    numeric_delta = (stored_df[numeric_columns] - expected_df[numeric_columns]).abs()
    max_numeric_abs_delta = float(numeric_delta.to_numpy().max())

roundtrip_contract_valid = bool(
    stored_df.columns.tolist() == expected_df.columns.tolist()
    and stored_df.index.equals(expected_df.index)
    and stored_df.shape == expected_df.shape
)

roundtrip_summary = pd.DataFrame(
    [
        {
            "stored_rows": int(len(stored_df)),
            "stored_columns": int(stored_df.shape[1]),
            "roundtrip_contract_valid": roundtrip_contract_valid,
            "max_numeric_abs_delta": max_numeric_abs_delta,
            "storage_target": storage_target,
        }
    ]
)

display(roundtrip_summary)
display(stored_df.head())

In [ ]:
offline_df = build_offline_store_frame(dataset=dataset)
entity_rows = build_entity_rows(dataset=dataset)
exported_path = export_offline_store(dataset=dataset, output_path=feast_export_path)

feast_roundtrip_ready = bool(
    len(offline_df) > 0 and len(entity_rows) > 0 and feast_runtime_status == "ready"
)

feast_summary = pd.DataFrame(
    [
        {
            "offline_rows": int(len(offline_df)),
            "offline_columns": int(offline_df.shape[1]),
            "entity_rows": int(len(entity_rows)),
            "entity_columns": int(entity_rows.shape[1]),
            "feast_roundtrip_ready": feast_roundtrip_ready,
            "feast_runtime_status": feast_runtime_status,
            "exported_path": str(exported_path),
        }
    ]
)

display(feast_summary)
display(offline_df.head())
display(entity_rows.head())

run_summary = {
    "runtime_lane": runtime_lane,
    "storage_backend": backend,
    "spot_id": spot_id,
    "dataset": dataset,
    "storage_target": storage_target,
    "raw_rows": int(len(raw_df)),
    "raw_columns": int(raw_df.shape[1]),
    "feature_rows": int(len(feature_df)),
    "feature_columns": int(feature_df.shape[1]),
    "validation_is_valid": bool(validation.is_valid),
    "validation_missing_columns": int(len(validation.missing_columns)),
    "validation_range_violations": int(len(validation.range_violations)),
    "stored_rows": int(len(stored_df)),
    "stored_columns": int(stored_df.shape[1]),
    "roundtrip_contract_valid": roundtrip_contract_valid,
    "max_numeric_abs_delta": max_numeric_abs_delta,
    "feast_roundtrip_ready": feast_roundtrip_ready,
    "offline_rows": int(len(offline_df)),
    "offline_columns": int(offline_df.shape[1]),
    "entity_rows": int(len(entity_rows)),
    "entity_columns": int(entity_rows.shape[1]),
    "feast_runtime_status": feast_runtime_status,
    "exported_path": str(exported_path),
}

summary_path.write_text(json.dumps(run_summary, indent=2))
display(pd.Series(run_summary, name="run").to_frame())
print(f"Wrote summary to {summary_path}")

In [ ]:
required_names = ["backend", "summary_dir", "run_summary"]
missing_names = [name for name in required_names if name not in globals()]

if missing_names:
    print(
        "Run the notebook from the first code cell before the comparison step. "
        f"Missing state: {', '.join(missing_names)}"
    )
else:
    counterpart_backend = {"s3": "bigquery", "bigquery": "s3"}.get(backend)
    counterpart_summary = None
    counterpart_summary_path = None
    if counterpart_backend is not None:
        counterpart_summary_path = (
            summary_dir / f"feature_pipeline_summary_{counterpart_backend}.json"
        )
        if counterpart_summary_path.exists():
            counterpart_summary = json.loads(counterpart_summary_path.read_text())

    stable_fields = [
        "spot_id",
        "dataset",
        "raw_rows",
        "raw_columns",
        "feature_rows",
        "feature_columns",
        "validation_is_valid",
        "validation_missing_columns",
        "validation_range_violations",
        "stored_rows",
        "stored_columns",
        "roundtrip_contract_valid",
        "max_numeric_abs_delta",
        "feast_roundtrip_ready",
        "offline_rows",
        "offline_columns",
        "entity_rows",
        "entity_columns",
    ]

    def values_match(field: str, current_value: object, other_value: object) -> bool:
        if field == "max_numeric_abs_delta":
            return abs(float(current_value) - float(other_value)) < 1e-12
        return current_value == other_value

    if counterpart_summary is None:
        if counterpart_summary_path is None:
            print("No counterpart backend is defined for this storage backend.")
        else:
            print(
                "No counterpart summary artifact found yet. "
                f"Run this same notebook with backend '{counterpart_backend}' and store the summary at {counterpart_summary_path}."
            )
    else:
        comparison_rows = [
            {
                "field": field,
                "current": run_summary.get(field),
                "counterpart": counterpart_summary.get(field),
                "matches": values_match(
                    field, run_summary.get(field), counterpart_summary.get(field)
                ),
            }
            for field in stable_fields
        ]
        comparison_df = pd.DataFrame(comparison_rows)
        display(comparison_df)

        expected_differences = pd.DataFrame(
            [
                {
                    "field": "storage_backend",
                    "current": run_summary.get("storage_backend"),
                    "counterpart": counterpart_summary.get("storage_backend"),
                },
                {
                    "field": "storage_target",
                    "current": run_summary.get("storage_target"),
                    "counterpart": counterpart_summary.get("storage_target"),
                },
                {
                    "field": "exported_path",
                    "current": run_summary.get("exported_path"),
                    "counterpart": counterpart_summary.get("exported_path"),
                },
            ]
        )
        display(expected_differences)
        print(
            "Reproducibility status across stable fields: "
            f"{'PASS' if bool(comparison_df['matches'].all()) else 'FAIL'}"
        )